# 00 — Origen, Obtención y Diccionario de Datos

## 1. Contexto del Problema

### ¿Por qué detectar anomalías en el consumo eléctrico de Ecuador?

Ecuador es un caso de estudio particularmente interesante para la detección de anomalías energéticas por varias razones:

1. **Alta dependencia hidroeléctrica (~70%)**: La central Coca Codo Sinclair (1,500 MW) y Paute (1,075 MW) dominan la generación. Cualquier reducción en caudales hídricos impacta directamente al sistema.

2. **Vulnerabilidad climática**: El fenómeno de El Niño/La Niña afecta los patrones de lluvia, creando períodos de sequía que reducen la capacidad de generación hidro.

3. **Crisis recientes documentadas**:
   - **Oct 2023 – Ene 2024**: Sequía severa, primeros racionamientos
   - **Abr – Jun 2024**: Apagones programados de hasta 8 horas
   - **Oct – Dic 2024**: Crisis máxima, racionamientos de hasta 14 horas diarias
   - **Dic 2025 – Ene 2026**: Recaída con aumento de generación fósil

4. **Relevancia práctica**: No existe una herramienta pública que detecte automáticamente estos eventos. Este proyecto busca llenar ese vacío.

### Marco teórico
La detección de anomalías en series temporales energéticas se apoya en:
- **Liu et al. (2008)** — "Isolation Forest" — algoritmo original que usamos
- **Chandola et al. (2009)** — "Anomaly Detection: A Survey" — taxonomía de métodos
- **Himeur et al. (2021)** — "Artificial intelligence based anomaly detection of energy consumption in buildings" — aplicación al sector energético

## 2. Fuentes de Datos

### 2.1 Ember — Global Electricity Data (Fuente principal)

| Atributo | Detalle |
|----------|---------|
| **Organización** | Ember (think tank energético global, Londres) |
| **URL** | https://ember-energy.org/data/monthly-electricity-data/ |
| **Licencia** | CC BY 4.0 (libre uso con atribución) |
| **Cobertura** | 85 países, datos mensuales |
| **Período Ecuador** | Enero 2019 — Enero 2026 (85 meses) |
| **Formato** | CSV (long format), ~500K filas globales |
| **Actualización** | Mensual |
| **Metodología** | Recopilación de fuentes nacionales oficiales (CENACE para Ecuador), validación cruzada con IEA y UN |
| **Descarga directa** | `storage.googleapis.com/emb-prod-bkt-publicdata/public-downloads/monthly_full_release_long_format.csv` |

**¿Por qué Ember y no CENACE directamente?** Los portales gubernamentales ecuatorianos (cenace.gob.ec, datosabiertos.gob.ec) presentan problemas de certificados SSL y timeouts frecuentes. Ember recopila datos oficiales del CENACE y los normaliza en formato estándar internacional.

### 2.2 Our World in Data — Energy Dataset (Complementaria)

| Atributo | Detalle |
|----------|---------|
| **Organización** | Our World in Data (Universidad de Oxford) |
| **URL** | https://ourworldindata.org/energy |
| **Licencia** | CC BY 4.0 |
| **Cobertura** | 200+ países, datos anuales |
| **Período Ecuador** | 1900 — 2024 |
| **Formato** | CSV, ~23K filas × 130 columnas |
| **Variables usadas** | Población, PIB, shares históricos de generación |
| **Propósito** | Contexto anual para enriquecer datos mensuales |

### 2.3 World Bank — Development Indicators (Complementaria)

| Atributo | Detalle |
|----------|---------|
| **Organización** | Banco Mundial |
| **URL** | https://data.worldbank.org/ |
| **API** | `api.worldbank.org/v2/country/EC/indicator/{ID}` |
| **Indicador usado** | `EG.USE.ELEC.KH.PC` (consumo per cápita kWh) |
| **Período** | 2000 — 2023 |

### Proceso de obtención
```
Ember (CSV global 500K filas)
    → Filtrar: Area == "Ecuador"
    → Pivotar: long → wide (generación por tipo, demanda, CO₂)
    → 85 filas × 14 columnas mensuales

Our World in Data (CSV global 23K filas)
    → Filtrar: country == "Ecuador"
    → Seleccionar: población, PIB, shares
    → Merge por año sobre datos mensuales

World Bank (JSON API)
    → GET indicador EG.USE.ELEC.KH.PC para Ecuador
    → Merge por año

Resultado final: 85 filas × 24 columnas
```

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../../data/raw/ecuador_electricity_real.csv', parse_dates=['fecha'])

print(f"{'='*60}")
print(f"DATASET: Sector Eléctrico de Ecuador")
print(f"{'='*60}")
print(f"Filas (meses):     {df.shape[0]}")
print(f"Columnas:          {df.shape[1]}")
print(f"Período:           {df['fecha'].min().strftime('%Y-%m')} → {df['fecha'].max().strftime('%Y-%m')}")
print(f"Granularidad:      Mensual")
print(f"Memoria:           {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
print(f"Fuente principal:  Ember Global Electricity Data")
print(f"Fuentes compl.:    Our World in Data, World Bank")
print(f"{'='*60}")

## 3. Diccionario de Datos Completo

Cada columna del dataset con su significado técnico, unidad y contexto.

In [ ]:
diccionario = pd.DataFrame([
    ['fecha', 'datetime', 'Ember', '-', 'Primer día del mes de medición',
     '100%', 'Índice temporal de la serie'],
    ['gen_bioenergy', 'float64', 'Ember', '% del mix', 
     'Generación eléctrica a partir de biomasa, biogás y residuos',
     '100%', 'Muy baja en Ecuador (<2%), apareció en 2025'],
    ['gen_clean', 'float64', 'Ember', '% del mix',
     'Generación limpia total = hidro + eólica + solar + bioenergía',
     '100%', 'Agregado: gen_hydro + gen_wind + gen_solar + gen_bioenergy'],
    ['gen_fossil', 'float64', 'Ember', '% del mix',
     'Generación fósil total = gas + petróleo + otros fósiles',
     '100%', 'Complemento de gen_clean. gen_clean + gen_fossil ≈ 100%'],
    ['gen_gas', 'float64', 'Ember', '% del mix',
     'Generación con gas natural',
     '100%', 'Ecuador tiene reservas limitadas de gas natural'],
    ['gen_hydro', 'float64', 'Ember', '% del mix',
     'Generación hidroeléctrica (Coca Codo Sinclair, Paute, etc.)',
     '100%', 'PRINCIPAL fuente (~70%). Variable clave del proyecto'],
    ['gen_other_fossil', 'float64', 'Ember', '% del mix',
     'Generación con petróleo (diésel, fuel oil) y otros combustibles fósiles',
     '100%', 'Plantas térmicas de emergencia, se activan en crisis'],
    ['gen_solar', 'float64', 'Ember', '% del mix',
     'Generación fotovoltaica',
     '100%', '≈0% — Ecuador tiene capacidad solar mínima instalada'],
    ['gen_total_generation', 'float64', 'Ember', 'TWh',
     'Generación eléctrica total del mes',
     '100%', 'Volumen absoluto, no porcentaje'],
    ['gen_wind', 'float64', 'Ember', '% del mix',
     'Generación eólica (parques eólicos de Loja, Galápagos)',
     '100%', '<1% del mix. Crecimiento incipiente'],
    ['demanda_twh', 'float64', 'Ember', 'TWh',
     'Demanda eléctrica total del país en el mes',
     '100%', 'Incluye pérdidas de transmisión y distribución'],
    ['co2_intensity_gco2_kwh', 'float64', 'Ember', 'gCO₂/kWh',
     'Gramos de CO₂ emitidos por cada kWh generado',
     '99%', 'Proxy directo del mix fósil. Sube cuando hidro baja'],
    ['importaciones_netas_twh', 'float64', 'Ember', 'TWh',
     'Energía importada (+) o exportada (-) con Colombia/Perú',
     '100%', 'Ecuador importa más durante crisis hídricas'],
    ['anio', 'int64', 'Calculado', '-', 'Año extraído de fecha', '100%', 'Clave para merge con OWID'],
    ['poblacion', 'float64', 'OWID', 'personas',
     'Población estimada de Ecuador',
     '85%', 'Anual. Nulo para años sin datos OWID publicados (2025+)'],
    ['pib_usd', 'float64', 'OWID', 'USD int.',
     'PIB nominal en dólares internacionales',
     '56%', 'Anual. Más nulos por retraso de publicación'],
    ['consumo_per_capita_kwh', 'float64', 'OWID', 'kWh/persona',
     'Consumo eléctrico por habitante',
     '85%', 'Indicador de desarrollo. Ecuador ≈ 1,500-1,900 kWh/cap'],
    ['share_hidro_pct', 'float64', 'OWID', '%',
     'Porcentaje anual de la generación que es hidroeléctrica',
     '85%', 'Referencia anual para comparar con datos mensuales'],
    ['share_fossil_pct', 'float64', 'OWID', '%',
     'Porcentaje anual de generación fósil',
     '85%', 'Complemento de share_hidro_pct'],
    ['share_renovable_pct', 'float64', 'OWID', '%',
     'Porcentaje anual de renovables (hidro + eólica + solar + bio)',
     '85%', 'Similar a share_hidro_pct en Ecuador por dominio hidro'],
], columns=['Variable', 'Tipo', 'Fuente', 'Unidad', 'Descripción', 'Completitud', 'Nota'])

# Mostrar todo
pd.set_option('display.max_colwidth', 80)
diccionario.style.set_properties(**{'text-align': 'left'}).set_table_styles(
    [{'selector': 'th', 'props': [('text-align', 'left')]}])

## 4. Verificación de integridad del dataset

In [ ]:
# Verificación completa
print("INTEGRIDAD DEL DATASET")
print("=" * 50)

# 1. Completitud temporal
rango = pd.date_range(df['fecha'].min(), df['fecha'].max(), freq='MS')
print(f"\n1. Completitud temporal:")
print(f"   Meses esperados: {len(rango)}")
print(f"   Meses presentes: {len(df)}")
print(f"   Meses faltantes: {len(set(rango) - set(df['fecha']))}")
print(f"   Estado: {'✓ COMPLETO' if len(set(rango) - set(df['fecha'])) == 0 else '✗ INCOMPLETO'}")

# 2. Nulos por fuente
print(f"\n2. Completitud por fuente:")
ember_cols = [c for c in df.columns if c.startswith('gen_') or c in ['demanda_twh', 'co2_intensity_gco2_kwh', 'importaciones_netas_twh']]
owid_cols = ['poblacion', 'pib_usd', 'consumo_per_capita_kwh', 'share_hidro_pct', 'share_fossil_pct', 'share_renovable_pct']
for name, cols in [('Ember (mensuales)', ember_cols), ('OWID (anuales)', [c for c in owid_cols if c in df.columns])]:
    existing = [c for c in cols if c in df.columns]
    if existing:
        completitud = df[existing].notna().mean().mean() * 100
        print(f"   {name}: {completitud:.1f}% completo")

# 3. Duplicados
print(f"\n3. Duplicados:")
print(f"   Filas duplicadas: {df.duplicated().sum()}")
print(f"   Fechas duplicadas: {df['fecha'].duplicated().sum()}")

# 4. Tipos de datos
print(f"\n4. Tipos correctos:")
print(f"   fecha → datetime: {'✓' if df['fecha'].dtype == 'datetime64[ns]' else '✗'}")
print(f"   numéricos → float: {'✓' if all(df[c].dtype in ['float64','int64'] for c in df.columns if c != 'fecha') else '✗'}")

# 5. Rango de valores
print(f"\n5. Rangos razonables:")
checks = [
    ('demanda_twh', 0, 10, 'TWh mensual entre 0-10'),
    ('gen_hydro', 0, 100, '% entre 0-100'),
    ('gen_fossil', 0, 100, '% entre 0-100'),
]
for col, lo, hi, desc in checks:
    if col in df.columns:
        ok = df[col].between(lo, hi).all()
        print(f"   {col}: {'✓' if ok else '✗'} ({desc})")

## 5. Resumen

- **85 meses** de datos reales (Ene 2019 — Ene 2026)
- **24 columnas** de 3 fuentes internacionales verificadas
- Variables mensuales de Ember: **100% completas**
- Variables anuales de OWID: **85% completas** (normal por retraso de publicación)
- **0 duplicados**, **0 meses faltantes**, **tipos correctos**
- Datos listos para análisis exploratorio y modelado

---
*Siguiente: 01 — Carga y Exploración*